In [22]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

# **Best-level OFI**

In [4]:
# Load the data
df = pd.read_csv('first_25000_rows.csv')
df['ts_recv'] = pd.to_datetime(df['ts_recv'])
df = df.sort_values('ts_recv').reset_index(drop=True) # make sure the time is in order, also do not have to consider the different symbols since only have AAPL

df['prev_bid_px'] = df['bid_px_00'].shift(1)
df['prev_ask_px'] = df['ask_px_00'].shift(1)
df['prev_bid_sz'] = df['bid_sz_00'].shift(1)
df['prev_ask_sz'] = df['ask_sz_00'].shift(1)

# Calculate Best-level OFI
def calculate_ofi(row):
    # Bid-side OFI
    if pd.isna(row['prev_bid_px']): # first row has no previous values
        ofi_bid = 0
    elif row['bid_px_00'] > row['prev_bid_px']: # price increases
        ofi_bid = row['bid_sz_00']
    elif row['bid_px_00'] == row['prev_bid_px']: # price keeps the same
        ofi_bid = row['bid_sz_00'] - row['prev_bid_sz']
    else: # price decreases
        ofi_bid = -row['bid_sz_00']

    # Ask-side OFI
    if pd.isna(row['prev_ask_px']):
        ofi_ask = 0
    elif row['ask_px_00'] > row['prev_ask_px']:
        ofi_ask = -row['ask_sz_00']
    elif row['ask_px_00'] == row['prev_ask_px']:
        ofi_ask = row['ask_sz_00'] - row['prev_ask_sz']
    else:
        ofi_ask = row['ask_sz_00']

    return pd.Series([ofi_bid, ofi_ask, ofi_bid - ofi_ask])

# Apply OFI calculation
df[['OFI_bid', 'OFI_ask', 'OFI']] = df.apply(calculate_ofi, axis=1)

df

,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,depth,price,size,...,bid_ct_09,ask_ct_09,symbol,prev_bid_px,prev_ask_px,prev_bid_sz,prev_ask_sz,OFI_bid,OFI_ask,OFI
0,2024-10-21 11:54:29.221230963+00:00,2024-10-21T11:54:29.221064336Z,10,2,38,C,B,1,233.62,2,...,2,1,AAPL,NaN,NaN,NaN,NaN,0.0,0.0,0.0
1,2024-10-21 11:54:29.223936626+00:00,2024-10-21T11:54:29.223769812Z,10,2,38,A,B,0,233.67,2,...,2,1,AAPL,233.67,233.74,139.0,200.0,2.0,0.0,2.0
2,2024-10-21 11:54:29.225196809+00:00,2024-10-21T11:54:29.225030400Z,10,2,38,A,B,0,233.67,3,...,2,1,AAPL,233.67,233.74,141.0,200.0,3.0,0.0,3.0
3,2024-10-21 11:54:29.712600612+00:00,2024-10-21T11:54:29.712434212Z,10,2,38,A,B,2,233.52,200,...,2,1,AAPL,233.67,233.74,144.0,200.0,0.0,0.0,0.0
4,2024-10-21 11:54:29.764839221+00:00,2024-10-21T11:54:29.764673165Z,10,2,38,C,B,2,233.52,200,...,2,1,AAPL,233.67,233.74,144.0,200.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,2024-10-21 13:04:16.583694069+00:00,2024-10-21T13:04:16.583527688Z,10,2,38,A,B,2,233.46,200,...,2,4,AAPL,233.51,233.61,1.0,20.0,0.0,0.0,0.0
4996,2024-10-21 13:04:17.976627074+00:00,2024-10-21T13:04:17.976461017Z,10,2,38,A,A,1,233.69,200,...,2,4,AAPL,233.51,233.61,1.0,20.0,0.0,0.0,0.0
4997,2024-10-21 13:04:20.085804687+00:00,2024-10-21T13:04:20.085638629Z,10,2,38,C,B,2,233.46,200,...,1,4,AAPL,233.51,233.61,1.0,20.0,0.0,0.0,0.0
4998,2024-10-21 13:04:20.085817362+00:00,2024-10-21T13:04:20.085651109Z,10,2,38,A,B,3,233.44,200,...,2,4,AAPL,233.51,233.61,1.0,20.0,0.0,0.0,0.0


In [41]:
# calculate accumulative OFIs during a chosen time interval (1 min)
df.set_index('ts_recv', inplace=True)
best_ofi_1min = df['OFI'].resample('1T', closed='right', label='right').sum() # data within time interval (t-1min,t]

best_ofi_1min

<ipython-input-41-0ecd41c6bdad>:3: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  best_ofi_1min = df['OFI'].resample('1T', closed='right', label='right').sum() # data within time interval (t-1min,t)


,OFI
ts_recv,
2024-10-21 11:55:00+00:00,-531.0
2024-10-21 11:56:00+00:00,-1217.0
2024-10-21 11:57:00+00:00,334.0
2024-10-21 11:58:00+00:00,960.0
2024-10-21 11:59:00+00:00,422.0
...,...
2024-10-21 13:01:00+00:00,-1007.0
2024-10-21 13:02:00+00:00,-1135.0
2024-10-21 13:03:00+00:00,-5575.0


# **Deeper-level OFI**

In [26]:
df = pd.read_csv('first_25000_rows.csv')
df['ts_recv'] = pd.to_datetime(df['ts_recv'])
df = df.sort_values('ts_recv').reset_index(drop=True)

N_LEVELS = 10

for level in range(N_LEVELS):
    lvl_str = f"{level:02d}"
    df[f'prev_bid_px_{lvl_str}'] = df[f'bid_px_{lvl_str}'].shift(1)
    df[f'prev_ask_px_{lvl_str}'] = df[f'ask_px_{lvl_str}'].shift(1)
    df[f'prev_bid_sz_{lvl_str}'] = df[f'bid_sz_{lvl_str}'].shift(1)
    df[f'prev_ask_sz_{lvl_str}'] = df[f'ask_sz_{lvl_str}'].shift(1)


def calculate_deeper_level_ofi(row, level):
    lvl_str = f"{level:02d}"

    # Current and previous values
    bid_px = row[f'bid_px_{lvl_str}']
    ask_px = row[f'ask_px_{lvl_str}']
    bid_sz = row[f'bid_sz_{lvl_str}']
    ask_sz = row[f'ask_sz_{lvl_str}']
    prev_bid_px = row.get(f'prev_bid_px_{lvl_str}', np.nan)
    prev_ask_px = row.get(f'prev_ask_px_{lvl_str}', np.nan)
    prev_bid_sz = row.get(f'prev_bid_sz_{lvl_str}', np.nan)
    prev_ask_sz = row.get(f'prev_ask_sz_{lvl_str}', np.nan)

    # Bid-side OFI
    if pd.isna(row[f'prev_bid_px_{lvl_str}']):
        ofi_bid = 0
    elif row[f'bid_px_{lvl_str}'] > row[f'prev_bid_px_{lvl_str}']:
        ofi_bid = row[f'bid_sz_{lvl_str}']
    elif row[f'bid_px_{lvl_str}'] == row[f'prev_bid_px_{lvl_str}']:
        ofi_bid = row[f'bid_sz_{lvl_str}'] - row[f'prev_bid_sz_{lvl_str}']
    else:
        ofi_bid = -row[f'bid_sz_{lvl_str}']

    # Ask-side OFI
    if pd.isna(row[f'prev_ask_px_{lvl_str}']):
        ofi_ask = 0
    elif row[f'ask_px_{lvl_str}'] > row[f'prev_ask_px_{lvl_str}']:
        ofi_ask = - row[f'ask_sz_{lvl_str}']
    elif row[f'ask_px_{lvl_str}'] == row[f'prev_ask_px_{lvl_str}']:
        ofi_ask = row[f'ask_sz_{lvl_str}'] - row[f'prev_ask_sz_{lvl_str}']
    else:
        ofi_ask = row[f'ask_sz_{lvl_str}']

    return ofi_bid - ofi_ask


# calculate OFIs for all levels
for level in range(N_LEVELS):
    df[f'OFI_level_{level}'] = df.apply(
        lambda row: calculate_deeper_level_ofi(row, level),
        axis=1
    )

TIME_WINDOW = '1T'  # 1-minute windows (adjust as needed)

df.set_index('ts_recv', inplace=True)
deeper_ofi_1min = df[[f'OFI_level_{level}' for level in range(N_LEVELS)]].resample(
    TIME_WINDOW, closed='right', label='right'
).sum()

# then we need to calculate the average order book depth
depth_columns = []
for level in range(N_LEVELS):
    level_str = f"{level:02d}"
    df[f'depth_level_{level}'] = (df[f'bid_sz_{level_str}'] + df[f'ask_sz_{level_str}']) # calculate for each order (row)
    depth_columns.append(f'depth_level_{level}')

# sum over a time interval (1 min)
windowed_data = df[depth_columns].resample(
    TIME_WINDOW, closed='right', label='right'
).agg(['sum', 'count'])

sum_depths = windowed_data.xs('sum', axis=1, level=1)
event_counts = windowed_data.xs('count', axis=1, level=1).mean(axis=1)  # ΔN(t)

Q = (sum_depths.div(2 * event_counts, axis=0)).mean(axis=1)

# scale the OFIs to get the final deeper level OFI
for level in range(N_LEVELS):
    deeper_ofi_1min[f'ofi_level_{level}'] = deeper_ofi_1min[f'OFI_level_{level}'] / Q

deeper_ofi_1min

<ipython-input-26-24adbeef04dc>:61: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  deeper_ofi_1min = df[[f'OFI_level_{level}' for level in range(N_LEVELS)]].resample(
<ipython-input-26-24adbeef04dc>:73: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  windowed_data = df[depth_columns].resample(


,OFI_level_0,OFI_level_1,OFI_level_2,OFI_level_3,OFI_level_4,OFI_level_5,OFI_level_6,OFI_level_7,OFI_level_8,OFI_level_9,ofi_level_0,ofi_level_1,ofi_level_2,ofi_level_3,ofi_level_4,ofi_level_5,ofi_level_6,ofi_level_7,ofi_level_8,ofi_level_9
ts_recv,,,,,,,,,,,,,,,,,,,,
2024-10-21 11:55:00+00:00,-531.0,346.0,542.0,-156.0,-61.0,580.0,44.0,-247.0,677.0,590.0,-4.920801,3.206398,5.022739,-1.445659,-0.565290,5.374886,0.407750,-2.288960,6.273790,5.467557
2024-10-21 11:56:00+00:00,-1217.0,1767.0,1821.0,-236.0,-380.0,738.0,-789.0,161.0,1388.0,-580.0,-11.204577,16.268273,16.765435,-2.172786,-3.498553,6.794559,-7.264101,1.482282,12.778926,-5.339897
2024-10-21 11:57:00+00:00,334.0,-573.0,1897.0,-585.0,652.0,-272.0,-64.0,850.0,-144.0,253.0,3.047986,-5.229030,17.311465,-5.338538,5.949961,-2.482192,-0.584045,7.756851,-1.314102,2.308804
2024-10-21 11:58:00+00:00,960.0,653.0,335.0,205.0,-1765.0,986.0,-504.0,-379.0,1667.0,-235.0,8.510998,5.789251,2.969984,1.817453,-15.647824,8.741504,-4.468274,-3.360071,14.778993,-2.083421
2024-10-21 11:59:00+00:00,422.0,160.0,922.0,965.0,-414.0,-1010.0,1035.0,-1.0,813.0,2056.0,3.676812,1.394052,8.033225,8.407876,-3.607110,-8.799954,9.017774,-0.008713,7.083527,17.913569
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-10-21 13:01:00+00:00,-1007.0,795.0,-343.0,-2598.0,2395.0,-1808.0,2360.0,921.0,-1079.0,629.0,-5.452230,4.304392,-1.857115,-14.066429,12.967320,-9.789109,12.777819,4.986598,-5.842062,3.405614
2024-10-21 13:02:00+00:00,-1135.0,-3698.0,-2878.0,1502.0,321.0,0.0,1230.0,901.0,-927.0,2224.0,-6.035417,-19.664291,-15.303902,7.986957,1.706933,0.000000,6.540584,4.791110,-4.929367,11.826226
2024-10-21 13:03:00+00:00,-5575.0,-4875.0,-5071.0,-138.0,-254.0,-1206.0,-1438.0,-1852.0,3267.0,-2068.0,-41.518680,-36.305572,-37.765242,-1.027727,-1.891613,-8.981440,-10.709213,-13.792394,24.330319,-15.401010


# **Integrated OFI**

In [34]:
ofi_matrix = deeper_ofi_1min[[f'ofi_level_{level}' for level in range(N_LEVELS)]]

# perform PCA to get first principal component
pca = PCA(n_components=1)
integrated_ofi_values = pca.fit_transform(ofi_matrix)

# normalize by L1 norm (sum to 1)
weights = pca.components_[0]
normalized_weights = weights / np.sum(np.abs(weights))

integrated_ofi = (ofi_matrix * normalized_weights).sum(axis=1)

integrated_ofi_1min = pd.DataFrame(
    data=integrated_ofi.values,
    index=ofi_matrix.index,
    columns=['integrated_ofi']
)

integrated_ofi_1min

,integrated_ofi
ts_recv,
2024-10-21 11:55:00+00:00,-0.847263
2024-10-21 11:56:00+00:00,-1.671067
2024-10-21 11:57:00+00:00,-6.781244
2024-10-21 11:58:00+00:00,5.386590
2024-10-21 11:59:00+00:00,-0.080479
...,...
2024-10-21 13:01:00+00:00,-9.101442
2024-10-21 13:02:00+00:00,4.544981
2024-10-21 13:03:00+00:00,7.671179


# **Cross-Asset OFI**

To construct the Cross-Asset OFI, we need OFIs for different tickers (multiple stocks), and explore the cross impact of OFIs by adding both the self term and the cross terms to the model (no matter it is best-level OFI or integrated OFI) and applying LASSO to solve it.

For the code, we may simply divide the big dataframe (consisting many stocks) into many small dataframes with only one stock. And then just loop all dataframes and apply previous codes to calculate different type of OFIs for all stocks.